In [3]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from datetime import datetime
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from scipy.stats import linregress
from pathlib import Path
from abc import ABCMeta, abstractmethod
from time import time

In [60]:
sys.path.append(os.path.abspath('..'))
from configs.config import *
from src.util import Logger, Util

In [5]:
# import importlib
# import configs.config
# importlib.reload(configs.config)
# from configs.config import *

In [6]:
pd.set_option("display.max_columns",200)
pd.set_option("display.max_rows", 500)

# 基底クラス

In [7]:
def decorate(s: str, decoration=None):
    if decoration is None:
        decoration = '★' * 20

    return ' '.join([decoration, str(s), decoration])

class Timer:
    def __init__(self, logger=None, format_str='{:.3f}[s]', prefix=None, suffix=None, sep=' ', verbose=0):

        if prefix: format_str = str(prefix) + sep + format_str
        if suffix: format_str = format_str + sep + str(suffix)
        self.format_str = format_str
        self.logger = logger
        self.start = None
        self.end = None
        self.verbose = verbose

    @property
    def duration(self):
        if self.end is None:
            return 0
        return self.end - self.start

    def __enter__(self):
        self.start = time()

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time()
        if self.verbose is None:
            return
        out_str = self.format_str.format(self.duration)
        if self.logger:
            self.logger.info(out_str)
        else:
            print(out_str)

In [8]:
class FeatureBase(metaclass=ABCMeta):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        self.use_cache = use_cache
        self.name = self.__class__.__name__
        self.cache_dir = Path(DIR_FEATURE)
        self.logger = logger
        self.seve_cache = save_cache
        self.use_cols = None
        self.key_column = None
    
    # 共通のキー整形 & 重複チェック
    def enforce_key_integrity(self, df: pd.DataFrame) -> pd.DataFrame:
        for key in self.key_column:
            if key not in df.columns:
                raise KeyError(f"{self.name}: キーカラム '{key}' が存在しません")
        assert ~df[self.key_column].duplicated().any(), f"{self.name}: 主キー {self.key_column} に重複があります"
    
    @abstractmethod
    def _create_feature(self) -> pd.DataFrame:
        """
        特徴量生成の実装をサブクラスで定義する必要があります。
        :return: pd.DataFrame 生成された特徴量
        """
        raise NotImplementedError()

    # 特徴量生成処理
    def create_feature(self) -> pd.DataFrame:

        # クラス名.pkl
        file_name = os.path.join(self.cache_dir, f"{self.name}.pkl")

        # キャッシュを使う & ファイルがあるなら読み出し
        if os.path.isfile(str(file_name)) and self.use_cache:
            feature = pd.read_pickle(file_name)

        # 変換処理を実行
        else:
            # train/testの区別なく変換処理を実行
            feature = self._create_feature()

            # 主キーチェック
            if self.key_column is not None:
                self.enforce_key_integrity(feature)

            # 保存する場合
            if self.seve_cache:
                feature.to_pickle(file_name)

        return feature

In [34]:
def one_hot_encode(df, col, drop_col=True):
    """
    特定の列に対してOne-Hotエンコーディングを適用します。
    
    :param df: pd.DataFrame 対象のDataFrame
    :param col: str エンコードする列名
    :return: pd.DataFrame エンコードされたDataFrame
    """
    # Initialize OneHotEncoder
    encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

    # Fit and transform the specified column
    encoded = encoder.fit_transform(df[[col]])

    # Convert the encoded array to a DataFrame
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out([col]))

    # concat
    df_concat = pd.concat([df, encoded_df], axis=1)

    # drop 
    if drop_col:
        df_concat.drop(columns=col, inplace=True)

    return df_concat

# 継承クラス

In [49]:
class Key(FeatureBase):
    """
    TrainFeatureクラスは、train.csvデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号', 'category']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        train.csvデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # train.csvデータを読み込む
        df_train = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_train.pkl'))
        df_test = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_test.pkl'))
        df_Key = pd.concat([df_train, df_test], ignore_index=True)[self.key_column]

        return df_Key

In [27]:
class Target(FeatureBase):
    """
    Targetクラスは、ターゲットデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号', 'category']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        ターゲットデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成されたターゲットデータを含むDataFrame。
        """
        # ターゲットデータを読み込む
        df_train = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_train.pkl'))

        # 必要なカラムを選択
        df_target = df_train[['社員番号', 'category', 'target']]

        # 主キーとターゲット列を含むDataFrameを返す
        return df_target

In [46]:
class CategoryFeature(FeatureBase):
    """
    TrainFeatureクラスは、train.csvデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['category']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        train.csvデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # train.csvデータを読み込む
        df_train = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_train.pkl'))
        df_test = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_test.pkl'))
        df_all = pd.concat([df_train, df_test], ignore_index=True)

        df_category_feature = df_all.copy().drop_duplicates('category')[self.key_column]

        # One-hotエンコーディング
        df_category_feature = one_hot_encode(df_category_feature, 'category', False)

        # 主キーとターゲット列を含むDataFrameを返す
        return df_category_feature

In [12]:
class CareerFeature(FeatureBase):
    """
    CareerBlockクラスは、キャリアデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        キャリアデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 前処理済みのキャリアデータを読み込む
        df_career = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_career.pkl"))

        df_career_feature = df_career.copy()

        # 特徴量例: キャリア関連の質問に対するポジティブな回答数
        df_career_feature['positive_responses'] = df_career_feature.iloc[:, 1:].apply(lambda row: (row == 1).sum(), axis=1)

        # 特徴量例: キャリア関連の質問に対するネガティブな回答数
        df_career_feature['negative_responses'] = df_career_feature.iloc[:, 1:].apply(lambda row: (row == 0).sum(), axis=1)

        # 特徴量例: ポジティブな回答の割合
        df_career_feature['positive_ratio'] = df_career_feature['positive_responses'] / (df_career_feature.shape[1] - 1)

        # 主キーと新しい特徴量を含むDataFrameを返す
        return df_career_feature

In [ ]:
   
class UdemyActivityFeature(FeatureBase):
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:

        # 前処理済みのUdemy活動データを読み込む
        df_udemy = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_udemy_activity.pkl"))

        # クイズ判定
        df_udemy["is_quiz"] = df_udemy["レクチャーもしくはクイズ"]=="Quiz"

        # 基本統計量の集計
        df_udemy_feature = df_udemy.groupby(self.key_column).agg(
            count_コースID=('コースID', 'count'),
            nunique_コースID=('コースID', 'nunique'),
            nunique_コースタイトル=('コースタイトル', 'nunique'),
            nunique_コースカテゴリ=('コースカテゴリー', 'nunique'),
            nunique_学習日数=('開始日', pd.Series.nunique),
            sum_マーク済み修了=('マーク済み修了', 'sum'),
            mean_推定完了率=('推定完了率%', 'mean'),
            min_開始日=('開始日', 'min'),
            max_開始日=('開始日', 'max'),
            rate_Quiz=('is_quiz', 'mean'),
        ).reset_index()

        # 学習スパン（日数）
        df_udemy_feature["learning_span"] = (df_udemy_feature["max_開始日"] - df_udemy_feature["min_開始日"]).dt.days
        # 日付型を数値型に変換
        df_udemy_feature["min_開始日"] = df_udemy_feature["min_開始日"].apply(lambda x: float(datetime.strftime(x, format='%Y%m%d')))
        df_udemy_feature["max_開始日"] = df_udemy_feature["max_開始日"].apply(lambda x: float(datetime.strftime(x, format='%Y%m%d')))

        # クイズスコアの集計
        df_quiz = df_udemy[df_udemy["is_quiz"]].copy()
        df_quiz_stats = df_quiz.groupby(self.key_column).agg(
            count_クイズ=('最終結果（クイズの場合）', 'count'),
            mean_クイズスコア=('最終結果（クイズの場合）', 'mean'),
        ).reset_index()

        # 結合
        df_udemy_feature = df_udemy_feature.merge(df_quiz_stats, on=self.key_column, how='left')

        return df_udemy_feature

In [14]:
class DxFeature(FeatureBase):
    """
    DxFeatureクラスは、DX関連のデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        DX関連データを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 前処理済みのDXデータを読み込む
        df_dx = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_dx.pkl"))

        # 特徴量例: 各社員の研修参加回数
        df_dx_feature = df_dx.groupby(self.key_column).agg(
            total_training_count=('研修名', 'count'),
        ).reset_index()

        # 特徴量例: 各社員のユニークな研修カテゴリ数
        # df_dx_feature['unique_training_categories'] = dx_data.groupby(self.key_column)['研修カテゴリ'].transform('nunique')

        return df_dx_feature
    

In [15]:
class HrFeature(FeatureBase):
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger=None)
        self.key_column = ['社員番号']

    def _create_feature(self) -> pd.DataFrame:

        df_hr = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_hr.pkl'))

        # 実施期間を算出（日数）
        df_hr['研修日数'] = (df_hr['実施終了日'] - df_hr['実施開始日']).dt.days + 1
        df_hr['研修日数'] = df_hr['研修日数'].fillna(1).clip(lower=1)

        # 特徴量作成
        df_hr_feature = df_hr.groupby("社員番号").agg(
            n_hr_total=("研修名", "count"),
            n_hr_unique_program=("研修名", "nunique"),
            n_hr_unique_category=("カテゴリ", "nunique"),
            first_hr_date=("実施開始日", "min"),
            last_hr_date=("実施終了日", "max"),
            n_hr_days=("研修日数", "sum"),
        ).reset_index()

        # 活動期間（最終日 - 初日）
        df_hr_feature["hr_active_days"] = (df_hr_feature["last_hr_date"] - df_hr_feature["first_hr_date"]).dt.days
        df_hr_feature.drop(["first_hr_date", "last_hr_date"], axis=1, inplace=True)

        return df_hr_feature

In [16]:
class OvertimeWorkByMonthFeature(FeatureBase):
    """
    OvertimeWorkFeatureクラスは、月ごとの残業データを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        残業データを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 残業データを読み込む
        df_overtime = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_overtime_work_by_month.pkl"))

        # 特徴量例: 各社員の月ごとの平均残業時間
        df_overtime_feature = df_overtime.groupby(self.key_column).agg(
            avg_overtime_hours=('hours', 'mean'),
            max_overtime_hours=('hours', 'max'),
            min_overtime_hours=('hours', 'min'),
            total_overtime_hours=('hours', 'sum'),
        ).reset_index()

        return df_overtime_feature

In [17]:
class PositionHistoryFeature(FeatureBase):
    """
    PositionHistoryFeatureクラスは、役職履歴データを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        役職履歴データを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 役職履歴データを読み込む
        df_position_history = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_position_history.pkl"))

        # 特徴量例: 各社員の役職変更回数
        df_position_history_feature = df_position_history.groupby(self.key_column).agg(
            position_change_count=('役職', 'nunique'),
            first_position=('役職', 'first'),
            last_position=('役職', 'last'),
        ).reset_index()

        # one-hotエンコーディング(OneHotEncoder)
        df_position_history_feature = one_hot_encode(df_position_history_feature, 'first_position')
        df_position_history_feature = one_hot_encode(df_position_history_feature, 'last_position')

        return df_position_history_feature

# 処理実行

In [83]:
def run_blocks(feature_blocks):
    print('start run blocks...')
    with Timer(prefix='run test'):
        for block in feature_blocks:
            with Timer(prefix='\t- {}'.format(str(block))):
                feature = block.create_feature()

In [84]:
feature_blocks = [
    Key(use_cache=True, save_cache=False, logger=None),
	Target(use_cache=True, save_cache=False, logger=None),
    CategoryFeature(use_cache=True, save_cache=False, logger=None),
	CareerFeature(use_cache=True, save_cache=False, logger=None),
	UdemyActivityFeature(use_cache=False, save_cache=True, logger=None),
	DxFeature(use_cache=True, save_cache=False, logger=None),
	HrFeature(use_cache=True, save_cache=False, logger=None),
	OvertimeWorkByMonthFeature(use_cache=True, save_cache=False, logger=None),
	PositionHistoryFeature(use_cache=True, save_cache=False, logger=None),
]

In [85]:
run_blocks(feature_blocks)

start run blocks...
	- <__main__.Key object at 0x0000017D58AC81F0> 0.024[s]
	- <__main__.Target object at 0x0000017D58AC8820> 0.013[s]
	- <__main__.CategoryFeature object at 0x0000017D58AC8F70> 0.007[s]
	- <__main__.CareerFeature object at 0x0000017D58AC89D0> 0.024[s]
	- <__main__.UdemyActivityFeature object at 0x0000017D53F19F40> 0.768[s]
	- <__main__.DxFeature object at 0x0000017D53F19A60> 0.028[s]
	- <__main__.HrFeature object at 0x0000017D53F19FD0> 0.025[s]
	- <__main__.OvertimeWorkByMonthFeature object at 0x0000017D53F19760> 0.020[s]
	- <__main__.PositionHistoryFeature object at 0x0000017D53F19F10> 0.021[s]
run test 0.931[s]


In [86]:
Util.load_feature('UdemyActivityFeature')

,社員番号,count_コースID,nunique_コースID,nunique_コースタイトル,nunique_コースカテゴリ,nunique_学習日数,sum_マーク済み修了,mean_推定完了率,min_開始日,max_開始日,rate_Quiz,learning_span,count_クイズ,mean_クイズスコア
0,-1sqs0GXzpPJuAVKHUUFgg==,32,4,1,1,24,29,93.807187,20220408.0,20240829.0,0.343750,874,11.0,54.545455
1,-2Sq3E0WkZj8pL7jxdL3Cg==,194,5,5,5,194,192,99.678646,20231211.0,20240919.0,0.010309,282,0.0,NaN
2,-4jh26kLzkU8JFQwdeQU9w==,2,2,1,1,2,1,52.275000,20221111.0,20230529.0,0.000000,199,NaN,NaN
3,-4taxxVbT1nU-J5fHWmDfQ==,316,36,33,22,295,244,85.585446,20220402.0,20250424.0,0.018987,1118,4.0,72.500000
4,-5W_JQCSTAYe2gGJMuT4_w==,146,17,11,9,140,136,95.579932,20220718.0,20250623.0,0.006849,1070,1.0,100.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2227,zow_7-qpjek7cWzX4-70fg==,224,14,12,8,221,212,96.648259,20230731.0,20250614.0,0.000000,684,NaN,NaN
2228,zs-xCihHPKBivyOIXncfzQ==,125,11,11,11,113,101,89.829680,20220517.0,20250429.0,0.032000,1077,4.0,87.500000
2229,zuplFpzBoM4c1dFy5HPXqg==,62,14,2,2,57,62,100.000000,20220630.0,20240927.0,0.000000,819,NaN,NaN
2230,zwcjIiu_sqUs8akLOfuYKA==,84,11,6,4,83,77,95.855119,20220617.0,20230704.0,0.059524,381,5.0,100.000000


In [62]:
Util.load_feature('UdemyActivityFeature').dtypes

社員番号                       object
count_コースID                 int64
nunique_コースID               int64
nunique_コースタイトル             int64
nunique_コースカテゴリ             int64
nunique_学習日数                int64
sum_マーク済み修了                 int32
mean_推定完了率                float64
min_開始日            datetime64[ns]
max_終了日            datetime64[ns]
rate_Quiz                 float64
learning_span             float64
count_クイズ                 float64
mean_クイズスコア               float64
dtype: object